# Goal 2 — Train Armenian SentencePiece Tokenizers

**Input:** cleaned CC-100 Armenian corpus (`hy_clean.txt`; not included in this repository because of size)

**Evaluation sample:** `data/sample/hy_sample_raw.txt`

**Output:** tokenizer files under `models/tokenizers/`


## 0. Setup & Imports

In [ ]:
# Install if needed (uncomment)
# !pip install sentencepiece tokenizers transformers tabulate matplotlib

import sentencepiece as spm
import os
import json
import time
import random
from pathlib import Path
from collections import Counter

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams['font.family'] = ['DejaVu Sans', 'Arial Unicode MS', 'sans-serif']
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

print(f"SentencePiece version: {spm.__version__}")

## 1. Configuration

### Why these choices?

```
LIBRARY CHOICE: SentencePiece vs HuggingFace tokenizers
-------------------------------------------------------
Considered:
  (a) SentencePiece (Google) — C++ core, Python wrapper
  (b) HuggingFace tokenizers — Rust core, Python wrapper

Chose SentencePiece because:
  - Both top performers in Goal 1 (XLM-R, mT5) use SentencePiece Unigram
  - SentencePiece has native, battle-tested Unigram implementation
  - HF tokenizers' Unigram trainer is less mature and less configurable
  - SentencePiece handles raw text directly (no pre-tokenization needed)
  - SentencePiece models (.model files) convert easily to HF format for grafting
  - BPE is also well-supported in SentencePiece for fair comparison

Tradeoff: HF tokenizers is faster for BPE training specifically,
but we prioritize Unigram quality and consistency across experiments.
```

```
VOCABULARY SIZES: 8k, 16k, 32k
--------------------------------
Considered: 4k, 8k, 16k, 32k, 48k

Chose 8k/16k/32k because:
  - 4k is too small: Armenian has ~38 base letters × upper/lower = ~76 characters,
    plus common suffixes, verb conjugations, noun declensions. 4k would barely
    cover the most frequent morphemes.
  - 8k is the minimum viable: covers the alphabet + common subwords + frequent
    whole words. Good for grafting onto models with limited vocab budget.
  - 16k is the sweet spot for a single-script tokenizer: enough to capture
    morphological patterns (Armenian is agglutinative — suffixes stack).
  - 32k matches LLaMA-2's total vocab size, giving us a reference point.
    Also useful if we want to fully replace (not extend) a tokenizer.
  - 48k+ is diminishing returns for a single language.

For grafting (Goal 3), we'll likely use 8k or 16k since we're ADDING
these to an existing vocab, not replacing it entirely.
```

```
PRE-TOKENIZATION: ArmTreeBank tokenizer?
-----------------------------------------
Considered using github.com/Armtreebank/Tokenizer as a pre-tokenization step
to split text into linguistically correct words before subword training.

Decided to SKIP because:
  - SentencePiece operates on raw text by design — it learns word boundaries
    from the data via its internal whitespace/character model.
  - Adding rule-based pre-tokenization constrains the subword model to
    respect word boundaries it may not need to respect (e.g., clitics,
    agglutinated suffixes might be better split differently).
  - XLM-R and mT5 (our best baselines) trained SentencePiece WITHOUT
    language-specific pre-tokenization — and they work well.
  - The ArmTreeBank tokenizer adds a dependency and complexity for
    marginal gain at the subword level.

If we had more time, it would be worth an ablation study comparing
with and without pre-tokenization. For a 3-day sprint, skip it.
```

```
TRAINING DATA SAMPLING: 5M sentences from 12M
-----------------------------------------------
Considered: full 12M, 5M sample, 2M sample

Chose 5M because:
  - SentencePiece loads ALL training sentences into memory at once.
  - 12M sentences × ~370 bytes avg = ~4.4GB raw text, plus internal
    data structures → could exceed 16GB RAM on M1 Pro.
  - SentencePiece's own docs recommend input_sentence_size=5M–10M
    for large corpora.
  - Subword statistics stabilize well before 5M sentences for a
    single language — diminishing returns beyond that.
  - 2M would also work but 5M gives more robust tail coverage
    (rare but valid Armenian words).
  - shuffle_input_sentence=True ensures the sample is representative.
```

```
CHARACTER COVERAGE: 0.9999 (not default 0.9995)
-------------------------------------------------
SentencePiece default is 0.9995, which drops the rarest 0.05% of characters.
For a multilingual model covering 100+ scripts, this makes sense.

For Armenian-only, we raise to 0.9999 because:
  - Armenian has a small character set (~80 chars including punctuation)
  - We don't want ANY Armenian letter to be mapped to <unk>
  - The 0.0001% we still drop catches true garbage (OCR artifacts,
    corrupted bytes) in CC-100 without losing real characters
  - Setting 1.0 risks including garbage characters in the vocab
```

In [ ]:
# =============================================================
# CONFIGURATION — edit paths here if running outside repo root
# =============================================================
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

# Full cleaned corpus is intentionally not committed. Set HY_CLEAN_PATH
# to its location, or place it at data/hy_clean.txt.
CORPUS_PATH = os.environ.get("HY_CLEAN_PATH", str(REPO_ROOT / "data" / "hy_clean.txt"))
EVAL_PATH = str(REPO_ROOT / "data" / "sample" / "hy_sample_raw.txt")
OUTPUT_DIR = str(REPO_ROOT / "models" / "tokenizers")

# Training parameters
SAMPLE_SIZE = 5_000_000               # sentences to sample for training
VOCAB_SIZES = [8000, 16000, 32000]    # vocab sizes to train
ALGORITHMS = ["unigram", "bpe"]       # both algorithms
CHARACTER_COVERAGE = 0.9999           # near-complete Armenian coverage
SEED = 42

# Special tokens — compatible with common LM conventions
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Corpus: {CORPUS_PATH}")
if os.path.exists(CORPUS_PATH):
    print(f"Corpus size: {os.path.getsize(CORPUS_PATH) / 1e9:.2f} GB")
else:
    print("Corpus missing. Set HY_CLEAN_PATH or place hy_clean.txt under data/.")
print(f"Eval sample: {EVAL_PATH}")
print(f"Output dir: {OUTPUT_DIR}/")


## 2. Prepare Training Data

SentencePiece can read directly from a text file, but we create a sampled version
to control memory usage and training time.

In [ ]:
def count_lines(filepath):
    """Fast line count using buffered reading."""
    count = 0
    with open(filepath, "rb") as f:
        for _ in f:
            count += 1
    return count

print("Counting lines in corpus (this takes ~30s for 4.5GB)...")
total_lines = count_lines(CORPUS_PATH)
print(f"Total lines: {total_lines:,}")

In [ ]:
# Reservoir sampling: pick exactly SAMPLE_SIZE lines uniformly at random
# without loading the entire file into memory.

SAMPLED_PATH = os.path.join(OUTPUT_DIR, "hy_train_sample.txt")

if os.path.exists(SAMPLED_PATH):
    existing_lines = count_lines(SAMPLED_PATH)
    print(f"Sampled file already exists with {existing_lines:,} lines. Skipping.")
else:
    print(f"Sampling {SAMPLE_SIZE:,} lines from {total_lines:,} (reservoir sampling)...")
    random.seed(SEED)
    
    # Reservoir sampling — O(N) time, O(SAMPLE_SIZE) memory
    reservoir = []
    with open(CORPUS_PATH, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            if i < SAMPLE_SIZE:
                reservoir.append(line)
            else:
                j = random.randint(0, i)
                if j < SAMPLE_SIZE:
                    reservoir[j] = line
            if (i + 1) % 2_000_000 == 0:
                print(f"  processed {i+1:,} lines...")
    
    # Shuffle to remove any ordering artifacts
    random.shuffle(reservoir)
    
    with open(SAMPLED_PATH, "w", encoding="utf-8") as f:
        for line in reservoir:
            f.write(line + "\n")
    
    sample_size_mb = os.path.getsize(SAMPLED_PATH) / 1e6
    print(f"Saved {len(reservoir):,} lines to {SAMPLED_PATH} ({sample_size_mb:.0f} MB)")
    del reservoir  # free memory

## 3. Train Tokenizers

We train 6 models: {Unigram, BPE} × {8k, 16k, 32k}.

```
KEY SENTENCEPIECE PARAMETERS EXPLAINED:
----------------------------------------
model_type:               'unigram' or 'bpe'
vocab_size:               target vocabulary size
input:                    path to training text file
model_prefix:             output path prefix (.model and .vocab files)
character_coverage:       fraction of characters to cover (0.9999)
input_sentence_size:      max sentences to load (we pre-sampled, so 0 = all)
shuffle_input_sentence:   shuffle before training (True)
num_threads:              CPU threads (M1 Pro has 8 performance cores)
byte_fallback:            encode unknown chars as UTF-8 bytes (True)
                          This is CRITICAL — ensures zero UNK rate by falling
                          back to byte tokens for any char not in vocab.
split_digits:             separate each digit (True, matches LLaMA convention)
allow_whitespace_only_pieces: needed for compatibility with some LMs
normalization_rule_name:  'nfkc' for Unicode normalization
                          (NFC would also work; NFKC is more aggressive
                          but standard in SentencePiece)

PARAMETERS WE CONSIDERED BUT LEFT AT DEFAULT:
----------------------------------------------
max_sentence_length:      default 4192 bytes. Our cleaning step already
                          removed very long lines, so no need to change.
seed_sentencepiece_size:  (Unigram only) initial vocab before pruning.
                          Default is 1M, which works well. Larger values
                          slow training but can capture rarer subwords.
shrinking_factor:         (Unigram only) how aggressively to prune per
                          iteration. Default 0.75 is well-tuned.
max_sentencepiece_length: default 16 bytes. Armenian chars are 2 bytes
                          each, so this allows tokens up to 8 Armenian
                          characters — reasonable for most words.
                          We increase to 24 to allow longer words.
```

In [ ]:
def train_tokenizer(algorithm, vocab_size, input_path, output_dir):
    """
    Train a SentencePiece tokenizer.
    
    Returns the path to the .model file.
    """
    model_prefix = os.path.join(
        output_dir, 
        f"hy_{algorithm}_{vocab_size // 1000}k"
    )
    
    # Skip if already trained
    if os.path.exists(model_prefix + ".model"):
        print(f"  Already exists: {model_prefix}.model — skipping")
        return model_prefix + ".model"
    
    print(f"  Training {algorithm.upper()} with vocab_size={vocab_size:,}...")
    t0 = time.time()
    
    spm.SentencePieceTrainer.train(
        input=input_path,
        model_prefix=model_prefix,
        model_type=algorithm,
        vocab_size=vocab_size,
        
        # Character and byte handling
        character_coverage=CHARACTER_COVERAGE,
        byte_fallback=True,
        
        # Normalization
        normalization_rule_name="nfkc",
        
        # Tokenization behavior
        split_digits=True,
        split_by_unicode_script=True,
        split_by_whitespace=True,
        split_by_number=True,
        allow_whitespace_only_pieces=True,
        
        # Token length — Armenian chars are 2 bytes each,
        # so 24 bytes = up to 12 Armenian characters per token
        max_sentencepiece_length=24,
        
        # Special tokens
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<s>",
        eos_piece="</s>",
        user_defined_symbols=["<mask>"],
        
        # Performance
        num_threads=8,
        shuffle_input_sentence=True,
        
        # Training control
        input_sentence_size=0,  # 0 = use all (we already sampled)
        train_extremely_large_corpus=False,
    )
    
    elapsed = time.time() - t0
    model_path = model_prefix + ".model"
    model_size_kb = os.path.getsize(model_path) / 1024
    print(f"  Done in {elapsed:.0f}s — model: {model_size_kb:.0f} KB")
    
    return model_path

In [ ]:
# Train all 6 configurations
trained_models = {}

for algo in ALGORITHMS:
    for vs in VOCAB_SIZES:
        key = f"{algo}_{vs // 1000}k"
        print(f"\n{'='*60}")
        print(f"Training: {key}")
        print(f"{'='*60}")
        model_path = train_tokenizer(algo, vs, SAMPLED_PATH, OUTPUT_DIR)
        trained_models[key] = model_path

print(f"\n\nAll {len(trained_models)} models trained!")
for key, path in trained_models.items():
    print(f"  {key}: {path}")

## 4. Load and Inspect Trained Tokenizers

In [ ]:
def load_sp_model(model_path):
    """Load a trained SentencePiece model."""
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

# Load all trained models
models = {}
for key, path in trained_models.items():
    models[key] = load_sp_model(path)
    sp = models[key]
    print(f"{key}: vocab_size={sp.get_piece_size():,}, "
          f"unk_id={sp.unk_id()}, bos_id={sp.bos_id()}, eos_id={sp.eos_id()}")

In [ ]:
# Inspect vocabulary composition for each model
def analyze_vocab(sp, name):
    """Analyze the vocabulary composition of a trained model."""
    total = sp.get_piece_size()
    
    armenian = 0
    byte_tokens = 0
    special = 0
    other = 0
    
    for i in range(total):
        piece = sp.id_to_piece(i)
        clean = piece.replace("\u2581", "")  # remove SentencePiece space marker
        
        if piece.startswith("<") and piece.endswith(">"):
            special += 1
        elif piece.startswith("<0x") and piece.endswith(">"):
            byte_tokens += 1
        elif clean and any("\u0530" <= c <= "\u058F" or "\uFB13" <= c <= "\uFB17" for c in clean):
            armenian += 1
        else:
            other += 1
    
    return {
        "name": name,
        "total": total,
        "armenian": armenian,
        "armenian_pct": round(armenian / total * 100, 1),
        "byte_tokens": byte_tokens,
        "special": special,
        "other": other,
    }

print("Vocabulary Composition:\n")
vocab_stats = []
for key, sp in models.items():
    stats = analyze_vocab(sp, key)
    vocab_stats.append(stats)
    print(f"{key}: {stats['armenian']:,} Armenian ({stats['armenian_pct']}%), "
          f"{stats['byte_tokens']} byte fallback, {stats['special']} special, "
          f"{stats['other']} other")

## 5. Evaluate on Held-Out Text

We compute the same metrics as Goal 1 (fertility, bytes/token, STRR, UNK rate)
on the held-out sample, then compare against the baseline tokenizers.

In [ ]:
import unicodedata
import re

def is_armenian_char(c):
    cp = ord(c)
    return (0x0530 <= cp <= 0x058F) or (0xFB13 <= cp <= 0xFB17)

def load_eval_lines(path, max_lines=None):
    """Load and lightly filter eval lines (same logic as Goal 1)."""
    lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            line = unicodedata.normalize("NFC", line.strip())
            if len(line) < 15:
                continue
            arm_count = sum(1 for c in line if is_armenian_char(c))
            alpha_count = sum(1 for c in line if c.isalpha())
            if alpha_count > 0 and arm_count >= 10 and (arm_count / alpha_count) >= 0.3:
                lines.append(line)
    return lines

eval_lines = load_eval_lines(EVAL_PATH)
print(f"Loaded {len(eval_lines):,} evaluation lines")
print(f"Total words: {sum(len(l.split()) for l in eval_lines):,}")

In [ ]:
def evaluate_sp_model(sp, lines, name):
    """
    Evaluate a SentencePiece model on held-out text.
    Returns dict with fertility metrics + UNK rate.
    """
    total_tokens = 0
    total_words = 0
    total_bytes = 0
    total_unks = 0
    single_token_words = 0
    word_fertilities = []
    
    unk_id = sp.unk_id()
    
    for line in lines:
        words = [w for w in line.split() if any(c.isalpha() for c in w)]
        if not words:
            continue
        
        # Full line encoding
        line_ids = sp.encode(line)
        total_tokens += len(line_ids)
        total_bytes += len(line.encode("utf-8"))
        total_words += len(words)
        total_unks += sum(1 for t in line_ids if t == unk_id)
        
        # Per-word
        for word in words:
            word_ids = sp.encode(word)
            n_tok = len(word_ids)
            word_fertilities.append(n_tok)
            if n_tok == 1:
                single_token_words += 1
    
    if total_words == 0 or total_tokens == 0:
        return None
    
    import statistics
    severe_frag = sum(1 for f in word_fertilities if f >= 5)
    
    return {
        "name": name,
        "total_tokens": total_tokens,
        "total_words": total_words,
        "fertility": round(total_tokens / total_words, 3),
        "bytes_per_token": round(total_bytes / total_tokens, 3),
        "strr": round(single_token_words / total_words, 4),
        "median_fertility": statistics.median(word_fertilities),
        "severe_frag_pct": round(severe_frag / total_words * 100, 2),
        "unk_rate": round(total_unks / total_tokens * 100, 4),
        "compression_ratio": round(total_bytes / total_tokens, 2),
    }

In [ ]:
# Evaluate all trained models
eval_results = []

for key, sp in models.items():
    print(f"Evaluating {key}...")
    result = evaluate_sp_model(sp, eval_lines, key)
    if result:
        eval_results.append(result)
        print(f"  fertility={result['fertility']:.2f}, "
              f"bytes/tok={result['bytes_per_token']:.2f}, "
              f"STRR={result['strr']:.3f}, "
              f"UNK={result['unk_rate']:.4f}%")

print(f"\nEvaluated {len(eval_results)} models.")

## 6. Results Comparison — Custom vs Baselines

Compare our trained tokenizers against the best and worst from Goal 1.

In [ ]:
# Goal 1 baselines for comparison
BASELINES = [
    {"name": "XLM-R (baseline best)", "fertility": 2.18, "bytes_per_token": 6.65,
     "strr": 0.4498, "severe_frag_pct": 7.3, "unk_rate": 0.0},
    {"name": "mT5 (baseline 2nd)", "fertility": 2.85, "bytes_per_token": 5.09,
     "strr": 0.1167, "severe_frag_pct": 8.4, "unk_rate": 0.0},
    {"name": "Gemma-2 (baseline mid)", "fertility": 4.49, "bytes_per_token": 3.23,
     "strr": 0.1004, "severe_frag_pct": 43.3, "unk_rate": 0.0},
    {"name": "GPT-2 (baseline worst)", "fertility": 14.26, "bytes_per_token": 1.02,
     "strr": 0.0042, "severe_frag_pct": 84.7, "unk_rate": 0.0},
]

# Combine for table
all_rows = eval_results + BASELINES
all_rows.sort(key=lambda x: x["fertility"])

headers = ["Tokenizer", "Fertility", "Bytes/Tok", "STRR", "Severe%", "UNK%"]
rows = []
for r in all_rows:
    rows.append([
        r["name"],
        f"{r['fertility']:.2f}",
        f"{r['bytes_per_token']:.2f}",
        f"{r['strr']:.3f}",
        f"{r['severe_frag_pct']:.1f}%",
        f"{r.get('unk_rate', 0.0):.4f}%",
    ])

if tabulate:
    print(tabulate(rows, headers=headers, tablefmt="grid"))
else:
    print(f"{'  '.join(h.ljust(20) for h in headers)}")
    print("-" * 120)
    for row in rows:
        print(f"{'  '.join(str(v).ljust(20) for v in row)}")

In [ ]:
# Visualization: fertility comparison
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    names = [r["name"] for r in all_rows]
    # Color: green for custom, blue for baselines
    colors = ["#1D9E75" if "baseline" not in r["name"] else "#3266AD" for r in all_rows]
    
    # Plot 1: Fertility
    ax = axes[0]
    ax.barh(names, [r["fertility"] for r in all_rows], color=colors)
    ax.set_xlabel("Fertility (tokens/word)")
    ax.set_title("Fertility (lower = better)")
    ax.invert_yaxis()
    
    # Plot 2: STRR
    ax = axes[1]
    ax.barh(names, [r["strr"] for r in all_rows], color=colors)
    ax.set_xlabel("STRR")
    ax.set_title("Single Token Retention Rate (higher = better)")
    ax.invert_yaxis()
    
    # Plot 3: Bytes/Token
    ax = axes[2]
    ax.barh(names, [r["bytes_per_token"] for r in all_rows], color=colors)
    ax.set_xlabel("Bytes/Token")
    ax.set_title("Compression (higher = better)")
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fertility_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Chart saved to {OUTPUT_DIR}/fertility_comparison.png")
else:
    print("matplotlib not available — skipping chart")

## 7. Sample Tokenization Demo

Show how each trained tokenizer segments example sentences.

In [ ]:
# Use the same sample sentence from Goal 1, plus a few more
# (Paste your own Armenian sentences here for inspection)
demo_sentences = [
    eval_lines[0] if eval_lines else "Test sentence",
]
# Add a few more representative lines
for line in eval_lines[1:200]:
    if 40 < len(line) < 100:
        demo_sentences.append(line)
    if len(demo_sentences) >= 3:
        break

for sent in demo_sentences:
    print(f"\nInput: {sent}")
    print("-" * 80)
    for key, sp in models.items():
        pieces = sp.encode(sent, out_type=str)
        print(f"  {key} ({len(pieces)} tokens): {pieces[:25]}{'...' if len(pieces) > 25 else ''}")

## 8. UNK Rate Analysis on Curated Test Set

Test with deliberately tricky Armenian text: rare words, loanwords,
proper nouns, numbers mixed with Armenian suffixes.

In [ ]:
# UNK analysis — check if byte_fallback is working correctly
# With byte_fallback=True, UNK rate should be 0% (all unknowns
# fall back to byte tokens, never to <unk>)

print("UNK Rate Check (should be 0.0000% with byte_fallback=True):\n")
for key, sp in models.items():
    total_toks = 0
    total_unks = 0
    unk_id = sp.unk_id()
    for line in eval_lines:
        ids = sp.encode(line)
        total_toks += len(ids)
        total_unks += sum(1 for t in ids if t == unk_id)
    unk_pct = total_unks / total_toks * 100 if total_toks > 0 else 0
    status = "PASS" if unk_pct == 0.0 else "WARN"
    print(f"  {key}: UNK rate = {unk_pct:.6f}%  [{status}]")

## 9. Select Best Model for Goal 3 (Grafting)

```
SELECTION CRITERIA:
-------------------
For grafting onto a small LM (Goal 3), we need to balance:

1. Fertility: lower is better (saves context window)
2. Vocab size: smaller is better for grafting (fewer new embeddings to learn)
3. STRR: higher means the model sees more complete words
4. UNK rate: must be 0% (byte_fallback handles this)

The tension is between fertility and vocab size:
- 32k vocab gives best fertility but adds 32k new embeddings to learn
- 8k vocab is easier to graft but fertility is worse
- 16k is likely the sweet spot

Algorithm choice (BPE vs Unigram) will depend on results above.
Unigram typically gives slightly better compression for agglutinative
languages because it considers the full token probability distribution,
while BPE greedily merges by frequency.
```

In [ ]:
# Rank models by a combined score
# Weighted: fertility (40%), STRR (30%), compression (20%), vocab_efficiency (10%)
# Smaller vocab is a bonus for grafting

print("Model Ranking for Grafting (Goal 3):\n")

scored = []
for r in eval_results:
    # Normalize metrics (lower fertility = better, higher STRR = better)
    fert_score = 1.0 / r["fertility"]  # inverse
    strr_score = r["strr"]
    comp_score = r["bytes_per_token"] / 10.0  # normalize
    
    # Vocab efficiency: prefer smaller vocabs (fewer embeddings to learn)
    vocab_size = int(r["name"].split("_")[1].replace("k", "")) * 1000
    vocab_score = 1.0 / (vocab_size / 8000)  # 8k=1.0, 16k=0.5, 32k=0.25
    
    combined = (0.4 * fert_score + 0.3 * strr_score + 
                0.2 * comp_score + 0.1 * vocab_score)
    scored.append((r["name"], combined, r["fertility"], r["strr"], vocab_size))

scored.sort(key=lambda x: -x[1])  # highest score first

for i, (name, score, fert, strr, vs) in enumerate(scored):
    marker = " <-- RECOMMENDED" if i == 0 else ""
    print(f"  {i+1}. {name}: score={score:.4f} "
          f"(fertility={fert:.2f}, STRR={strr:.3f}, vocab={vs:,}){marker}")

best_model_key = scored[0][0]
print(f"\nBest model for grafting: {best_model_key}")
print(f"Model file: {trained_models[best_model_key]}")

## 10. Export Best Model to HuggingFace Format

Convert the SentencePiece .model to HuggingFace tokenizer format
for seamless use in Goal 3 (grafting onto a transformer model).

```
WHY CONVERT TO HF FORMAT:
  - Goal 3 uses HuggingFace transformers for model surgery
  - The grafting code needs AutoTokenizer.from_pretrained() compatibility
  - HF format includes tokenizer.json, tokenizer_config.json, special_tokens_map.json
  - SentencePiece .model alone works but lacks the metadata HF expects
```

In [ ]:
from transformers import AutoTokenizer
import shutil

def export_to_hf(sp_model_path, output_dir, base_tokenizer="FacebookAI/xlm-roberta-base"):
    """
    Convert SentencePiece model to HuggingFace tokenizer directory.
    
    We use XLM-R as the base config because it's also SentencePiece Unigram,
    so the tokenizer_config.json structure is compatible.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Copy the .model file
    dest_model = os.path.join(output_dir, "sentencepiece.bpe.model")
    shutil.copy(sp_model_path, dest_model)
    
    # Load via HF to generate proper config files
    try:
        from transformers import XLMRobertaTokenizer
        tokenizer = XLMRobertaTokenizer(dest_model)
        tokenizer.save_pretrained(output_dir)
        print(f"Exported HF tokenizer to: {output_dir}/")
        print(f"Files: {os.listdir(output_dir)}")
        return True
    except Exception as e:
        print(f"HF export failed ({e}) — .model file is still usable directly")
        return False

# Export the best model
best_sp_path = trained_models[best_model_key]
hf_output_dir = os.path.join(OUTPUT_DIR, f"hf_{best_model_key}")
export_to_hf(best_sp_path, hf_output_dir)

## 11. Save All Results

In [ ]:
# Save evaluation results as JSON
results_path = os.path.join(OUTPUT_DIR, "goal2_eval_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump({
        "eval_corpus": EVAL_PATH,
        "training_corpus": CORPUS_PATH,
        "sample_size": SAMPLE_SIZE,
        "results": eval_results,
        "vocab_composition": vocab_stats,
        "best_model_for_grafting": best_model_key,
        "best_model_path": trained_models[best_model_key],
        "ranking": [{"name": s[0], "score": s[1], "fertility": s[2], 
                     "strr": s[3], "vocab_size": s[4]} for s in scored],
    }, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {results_path}")
print(f"\nTrained model files:")
for key, path in trained_models.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"  {key}: {path} ({size_kb:.0f} KB)")

print(f"\nHF export: {hf_output_dir}/")
print(f"\n--- Goal 2 complete! Ready for Goal 3 (grafting). ---")

---

## Summary of Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Library | SentencePiece | Used by both top Goal 1 tokenizers (XLM-R, mT5); best Unigram implementation |
| Algorithms | Both Unigram and BPE | Empirical comparison; Unigram expected to win for agglutinative Armenian |
| Vocab sizes | 8k, 16k, 32k | Covers grafting-friendly (8k) to standalone (32k) range |
| Training data | 5M sentence sample | Balances statistical coverage with M1 Pro memory constraints |
| Char coverage | 0.9999 | Near-complete; drops only true garbage while keeping all Armenian chars |
| byte_fallback | True | Guarantees 0% UNK rate — critical for robustness |
| Pre-tokenization | None (skip ArmTreeBank) | SentencePiece handles this internally; adds complexity for marginal gain |
| Normalization | NFKC | Standard for SentencePiece; consistent with XLM-R/mT5 baselines |
| max_sentencepiece_length | 24 bytes | Allows tokens up to 12 Armenian characters (2 bytes each) |
| split_digits | True | Matches LLaMA/modern LM convention; numbers handled separately |
| Special tokens | pad/unk/bos/eos/mask | Compatible with transformer LM conventions for grafting |